In [1]:
from transformer_lens import HookedTransformer
import torch
import torch.nn as nn
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from datasets import load_dataset
from torch.optim import Adam

In [2]:
model = HookedTransformer.from_pretrained("pythia-14m")
model.eval()
n_layers = model.cfg.n_layers
d_model = model.cfg.d_model

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-14m into HookedTransformer


In [3]:
device = next(model.parameters()).device
probes = nn.ModuleList([
    nn.Linear(d_model, d_model, bias=True)
    for _ in range(n_layers)
]).to(device)

for probe in probes:
    nn.init.eye_(probe.weight)
    nn.init.zeros_(probe.bias)

optimizer = Adam(probes.parameters(), lr=1e-3)

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:500]")
texts = [x for x in dataset["text"] if len(x.strip()) > 50][:200]

print(f"Training on {len(texts)} samples...")
probes.train()

for epoch in range(2):
    total_loss = 0
    for text in texts:
        tokens = model.to_tokens(text)
        if tokens.shape[1] < 3:
            continue

        with torch.no_grad():
            final_logits, cache = model.run_with_cache(tokens)

        loss = 0
        for i in range(n_layers):
            resid = cache[f"blocks.{i}.hook_resid_post"].squeeze(0)  # (seq, d_model)
            translated = probes[i](resid)
            translated_normed = model.ln_final(translated)
            layer_logits = model.unembed(translated_normed)  # (seq, vocab)

            # Train to match the final layer's distribution (KL divergence)
            final_probs = torch.softmax(final_logits.squeeze(0), dim=-1)
            log_layer_probs = torch.log_softmax(layer_logits, dim=-1)
            kl = (final_probs * (final_probs.log() - log_layer_probs)).sum(-1).mean()
            loss = loss + kl

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} loss: {total_loss/len(texts):.4f}")

probes.eval()
print("Done training!")

Training on 187 samples...
Epoch 1 loss: 33.2089
Epoch 2 loss: 35.2728
Done training!


In [4]:
def run_tuned_lens(prompt):
    tokens = model.to_str_tokens(prompt)
    token_ids = model.to_tokens(prompt)

    with torch.no_grad():
        _, cache = model.run_with_cache(token_ids)

    all_logits = []
    for i in range(n_layers):
        resid = cache[f"blocks.{i}.hook_resid_post"].squeeze(0)
        translated = probes[i](resid)
        translated_normed = model.ln_final(translated)
        logits = model.unembed(translated_normed)
        all_logits.append(logits)

    all_logits = torch.stack(all_logits)
    return tokens, all_logits

In [5]:
prompt = "The sky is blue and the grass is green."
tokens, all_logits = run_tuned_lens(prompt)

probs = torch.softmax(all_logits, dim=-1)
top1_probs = probs.max(dim=-1).values.detach().cpu().numpy()
top_ids = all_logits.argmax(dim=-1).detach().cpu()
top_strs = [[model.to_single_str_token(int(top_ids[l, s])) for s in range(len(tokens))]
            for l in range(n_layers)]
entropy = -(probs * probs.log().clamp(min=-1e9)).sum(-1).detach().cpu().numpy()

fig1 = go.Figure(go.Heatmap(
    z=top1_probs,
    x=tokens,
    y=[f"L{i}" for i in range(n_layers)],
    text=top_strs,
    texttemplate="%{text}",
    colorscale="Blues",
    colorbar=dict(title="Top-1 Prob"),
    zmin=0, zmax=1,
))
fig1.update_layout(
    title="Tuned Lens — Top Predicted Token (color = confidence)",
    xaxis_title="Input Token", yaxis_title="Layer", height=450
)
fig1.show()

fig2 = go.Figure(go.Heatmap(
    z=entropy,
    x=tokens,
    y=[f"L{i}" for i in range(n_layers)],
    colorscale="Reds_r",
    colorbar=dict(title="Entropy"),
))
fig2.update_layout(
    title="Tuned Lens — Prediction Entropy",
    xaxis_title="Input Token", yaxis_title="Layer", height=450
)
fig2.show()

In [ ]:
out = widgets.Output()
text_input = widgets.Text(
    value="The sky is blue and the grass is green.",
    description="Prompt:",
    layout=widgets.Layout(width="600px")
)
mode_toggle = widgets.ToggleButtons(
    options=["Top Token", "Entropy", "Top-1 Prob"],
    description="View:",
)

def update(_):
    with out:
        clear_output(wait=True)
        prompt = text_input.value
        if not prompt.strip():
            return

        tokens, all_logits = run_tuned_lens(prompt)
        probs = torch.softmax(all_logits, dim=-1)
        top1_probs = probs.max(dim=-1).values.detach().cpu().numpy()
        top_ids = all_logits.argmax(dim=-1).detach().cpu()
        top_strs = [[model.to_single_str_token(int(top_ids[l, s])) for s in range(len(tokens))]
                    for l in range(n_layers)]
        entropy = -(probs * probs.log().clamp(min=-1e9)).sum(-1).detach().cpu().numpy()
        mode = mode_toggle.value

        if mode == "Top Token":
            z, colorscale, title, colorbar = top1_probs, "Blues", "Top Predicted Token", "Top-1 Prob"
            text = top_strs
            texttemplate = "%{text}"
        elif mode == "Entropy":
            z, colorscale, title, colorbar = entropy, "Reds_r", "Prediction Entropy", "Entropy"
            text, texttemplate = None, None
        else:
            z, colorscale, title, colorbar = top1_probs, "Greens", "Top-1 Probability", "Prob"
            text, texttemplate = None, None

        fig = go.Figure(go.Heatmap(
            z=z, x=tokens, y=[f"L{i}" for i in range(n_layers)],
            text=text, texttemplate=texttemplate,
            colorscale=colorscale, colorbar=dict(title=colorbar),
            zmin=0 if mode != "Entropy" else None,
            zmax=1 if mode != "Entropy" else None,
        ))
        fig.update_layout(
            title=f"Tuned Lens — {title} — {prompt[:40]}{'...' if len(prompt) > 40 else ''}",
            xaxis_title="Input Token", yaxis_title="Layer", height=420
        )
        fig.show()

text_input.observe(update, names="value")
mode_toggle.observe(update, names="value")
display(text_input, mode_toggle, out)
update(None)

Text(value='The sky is blue and the grass is green.', description='Prompt:', layout=Layout(width='600px'))

ToggleButtons(description='View:', options=('Top Token', 'Entropy', 'Top-1 Prob'), value='Top Token')

Output()